# LangGraph vs Deep Agents — Comparison

Side-by-side analysis of two implementations of the same commercial assistant:

| | LangGraph | Deep Agents |
|---|---|---|
| File | `langgraph/agent/agent.py` + `state.py` | `deep_agents/agent/agent.py` |
| Approach | Low-level `StateGraph` | High-level `create_deep_agent()` |
| HITL | Manual `interrupt()` in graph node | Declarative `interrupt_on` dict |

## Sections
1. Code metrics
2. Architecture
3. LangSmith traces
4. Side-by-side behavioral test
5. Summary

> **Prerequisites:** run `langgraph/demo.ipynb` and `deep_agents/demo.ipynb` first
> to populate the LangSmith projects `lab-06-langgraph` and `lab-06-deep-agents`.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

# ── sys.path: lab root must be on path for namespace package imports ───────────
_cwd = Path().resolve()
_lab_root = _cwd.parent if _cwd.name in ("langgraph", "deep_agents") else _cwd
if str(_lab_root) not in sys.path:
    sys.path.insert(0, str(_lab_root))

load_dotenv(find_dotenv())

# ── LangSmith tracing for comparison runs ─────────────────────────────────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "lab-06-comparison"

# ── LangSmith project names from demo notebooks ───────────────────────────────
LG_PROJECT = "lab-06-langgraph"
DA_PROJECT = "lab-06-deep-agents"

# ── Provider / model ──────────────────────────────────────────────────────────
PROVIDER = "anthropic"        # "anthropic" | "openai" | "google"
MODEL    = "claude-sonnet-4-6"

print(f"Provider : {PROVIDER} / {MODEL}")
print(f"Lab root : {_lab_root}")

## 1 — Code Metrics

Counting non-blank, non-comment lines ("effective LOC") for each agent implementation.

In [ ]:
def loc(path: Path) -> dict:
    lines = path.read_text().splitlines()
    code = [
        line for line in lines
        if line.strip() and not line.strip().startswith("#") and line.strip() != '"""'
    ]
    return {"total": len(lines), "effective": len(code)}

files = {
    "LangGraph — agent.py"  : _lab_root / "langgraph/agent/agent.py",
    "LangGraph — state.py"  : _lab_root / "langgraph/agent/state.py",
    "Deep Agents — agent.py": _lab_root / "deep_agents/agent/agent.py",
}

print(f"{'File':<30} {'Total':>7} {'Effective':>10}")
print("-" * 50)
totals = {"LangGraph": 0, "Deep Agents": 0}
for label, path in files.items():
    m = loc(path)
    print(f"{label:<30} {m['total']:>7} {m['effective']:>10}")
    key = label.split("—")[0].strip()
    totals[key] += m["effective"]

print("-" * 50)
print(f"{'LangGraph total':<30} {'':>7} {totals['LangGraph']:>10}")
print(f"{'Deep Agents total':<30} {'':>7} {totals['Deep Agents']:>10}")
ratio = totals["LangGraph"] / totals["Deep Agents"]
print(f"\nRatio  LangGraph / Deep Agents : {ratio:.1f}×")

## 2 — Architecture

### 2.1 — LangGraph: explicit control flow

LangGraph exposes the full execution graph. Every node and edge is explicit:
- **`agent`** → calls LLM with tool bindings
- **`hitl`** → manually calls `interrupt()` after conditional check
- **`tools`** → executes tool calls via `ToolNode`

The graph can be visualised:

In [ ]:
from langgraph.agent.agent import create_agent as create_lg_agent

lg_agent = create_lg_agent(provider=PROVIDER, model=MODEL)
lg_agent.get_graph().print_ascii()

### 2.2 — HITL granularity

This is the most significant behavioral difference between the two implementations.

**LangGraph** — conditional HITL (in `_find_hitl_call`):
```python
# interrupt only when marking a lead as lost
if tc["name"] == "update_lead_status_tool" and tc["args"].get("new_status") == "lost":
    return tc
```

**Deep Agents** — declarative HITL (per-tool only, no argument inspection):
```python
HITL_TOOLS = {
    "generate_email_draft_tool": True,
    "update_lead_status_tool": True,   # ALL status changes
}
```

Consequence: marking a lead `prospect → qualified` triggers HITL in Deep Agents
but **not** in LangGraph. The declarative approach trades granularity for simplicity.

### 2.3 — Resume API

| | LangGraph | Deep Agents |
|---|---|---|
| Approve | `Command(resume={"decision": "approve"})` | `Command(resume={"decisions": [{"type": "approve"}]})` |
| Cancel | `Command(resume={"decision": "cancel", "feedback": "..."})` | `Command(resume={"decisions": [{"type": "reject"}]})` |
| Interrupt payload | `{"tool_name": ..., "tool_args": ..., "message": ...}` | `{"action_requests": [{"name": ..., "args": ...}]}` |

## 3 — LangSmith Traces

Fetches root-level runs from both demo projects and compares latency and token usage.

> Each scenario execution in the demo notebooks corresponds to one root run.

In [ ]:
from langsmith import Client

client = Client()

# Root runs = execution_order 1 (one per agent.invoke / agent.stream call)
lg_runs = [
    r for r in client.list_runs(project_name=LG_PROJECT)
    if r.parent_run_id is None
]
da_runs = [
    r for r in client.list_runs(project_name=DA_PROJECT)
    if r.parent_run_id is None
]

print(f"Root runs — LangGraph   : {len(lg_runs)}")
print(f"Root runs — Deep Agents : {len(da_runs)}")

In [ ]:
import pandas as pd


def runs_to_df(runs: list, label: str) -> pd.DataFrame:
    rows = []
    for r in runs:
        latency = None
        if r.end_time and r.start_time:
            latency = round((r.end_time - r.start_time).total_seconds(), 2)
        rows.append({
            "agent"       : label,
            "run_name"    : r.name,
            "status"      : r.status,
            "latency_s"   : latency,
            "total_tokens": r.total_tokens,
            "prompt_tokens": r.prompt_tokens,
            "completion_tokens": r.completion_tokens,
        })
    return pd.DataFrame(rows)


lg_df = runs_to_df(lg_runs, "LangGraph")
da_df = runs_to_df(da_runs, "Deep Agents")
all_df = pd.concat([lg_df, da_df], ignore_index=True)

print(all_df.to_string(index=False))

In [ ]:
# Summary stats — average latency and token usage per agent
cols = ["latency_s", "total_tokens", "prompt_tokens", "completion_tokens"]
summary = (
    all_df
    .groupby("agent")[cols]
    .mean()
    .round(1)
)
print("Average per root run:")
print(summary.to_string())

### LLM calls per scenario

For a finer view of how many LLM invocations each agent makes per scenario,
fetch child runs of type `"llm"` for each root run.

In [ ]:
def count_children(run, client: Client) -> dict:
    # trace_id == run.id for root runs — fetches all runs in the trace
    all_in_trace = list(client.list_runs(trace_id=str(run.id)))
    children = [r for r in all_in_trace if r.parent_run_id is not None]
    return {
        "llm_calls" : sum(1 for c in children if c.run_type == "llm"),
        "tool_calls": sum(1 for c in children if c.run_type == "tool"),
    }


# Only look at successful runs to avoid noise from API errors
print(f"{'Agent':<14} {'Status':<10} {'LLM calls':>10} {'Tool calls':>11}")
print("-" * 48)

for label, runs in (("LangGraph", lg_runs), ("Deep Agents", da_runs)):
    for r in runs:
        if r.status != "success":
            continue
        counts = count_children(r, client)
        print(
            f"{label:<14} {r.status:<10}"
            f" {counts['llm_calls']:>10} {counts['tool_calls']:>11}"
        )

## 4 — Side-by-side Behavioral Test

Run identical prompts through both agents and compare the responses.
Using `get_pipeline_stats_tool` — no HITL, fast execution.

In [ ]:
from langchain_core.messages import HumanMessage

from deep_agents.agent.agent import create_agent as create_da_agent

lg_agent = create_lg_agent(provider=PROVIDER, model=MODEL)
da_agent = create_da_agent(provider=PROVIDER, model=MODEL)

PROMPT = "Give me the pipeline statistics."

lg_result = lg_agent.invoke(
    {"messages": [HumanMessage(content=PROMPT)]},
    {"configurable": {"thread_id": "cmp-lg-stats"}},
)
da_result = da_agent.invoke(
    {"messages": [HumanMessage(content=PROMPT)]},
    {"configurable": {"thread_id": "cmp-da-stats"}},
)


def last_text(result):
    for msg in reversed(result["messages"]):
        content = msg.content
        if isinstance(content, list):
            content = " ".join(
                b.get("text", "") for b in content
                if isinstance(b, dict) and b.get("type") == "text"
            )
        if content:
            return content
    return ""


print("=" * 60)
print(f"Prompt: {PROMPT}")
print("=" * 60)

print("\n── LangGraph ──")
print(last_text(lg_result))

print("\n── Deep Agents ──")
print(last_text(da_result))

## 5 — Summary

| Criterion | LangGraph | Deep Agents |
|---|---|---|
| **Lines of code** | ~150 (agent + state) | ~70 |
| **Control flow** | Explicit `StateGraph` — fully visible | Implicit — hidden inside `create_deep_agent` |
| **Graph visualization** | ✅ `get_graph().print_ascii()` | ⚠️ Internal only |
| **HITL granularity** | Per-argument conditional (`new_status='lost'`) | Per-tool only (all calls) |
| **Resume API** | `{"decision": "approve/cancel"}` | `{"decisions": [{"type": "approve/reject"}]}` |
| **Interrupt payload** | Custom dict (`tool_name`, `tool_args`, `message`) | Fixed schema (`action_requests`) |
| **Extensibility** | High — add custom nodes, edges, state fields | Medium — constrained by framework |
| **Debuggability** | High — explicit graph + LangSmith traces | Medium — LangSmith traces only |
| **Dev time** | Higher — more boilerplate | Lower — faster to write |
| **Token usage** | Similar | Similar |
| **Latency** | Similar | Similar |

### When to choose which

**LangGraph** — when you need:
- Fine-grained HITL (per-argument conditions)
- Custom state fields beyond messages
- Complex conditional routing between nodes
- Maximum transparency for debugging

**Deep Agents** — when you need:
- Fast prototyping
- Standard tool-calling + HITL pattern
- Minimal boilerplate for straightforward agents